# 02 · Segmentar Wine Quality sin etiquetas

**Módulo 5 · Sesión 12** — Aprendizaje no supervisado

## Objetivos

El notebook 01 mostró qué supone cada algoritmo sobre datos de dos dimensiones, donde
se puede mirar. Aquí no se puede: Wine Quality tiene 11 variables fisicoquímicas, y el
flujo completo de un análisis de clustering real es la única forma de saber qué hay:

1. **Preparar** los datos: quitar duplicados, estandarizar, y medir qué pasa si no se
   estandariza.
2. Elegir $k$ con **codo, silueta y Davies-Bouldin**, ver que no siempre coinciden, y
   comparar contra una **referencia nula** para saber si la estructura encontrada es real.
3. **Interpretar** los grupos con el perfil de cada uno en unidades originales, y
   validarlos *a posteriori* contra dos etiquetas que el algoritmo no vio: `tipo` y
   `quality`.
4. Buscar estructura de **segundo nivel** dentro de un grupo.
5. Comparar con el **jerárquico** (y ver qué hacen los distintos *linkages* con datos
   reales) y con **DBSCAN**, que en 11 dimensiones sirve más como **detector de
   anomalías** que como algoritmo de grupos; 🔵 contrastarlo con Isolation Forest.
6. Medir la **estabilidad** de la partición.

La teoría está en `01-clustering.md` y `02-validacion-clusters.md`. Los datos son los
mismos del módulo 4 (`../datos/wine-quality.csv`), sin duplicados, como allí.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scipy`, `scikit-learn`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-5-no-supervisado-deep-learning/notebooks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, normalized_mutual_info_score, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Datos: sin duplicados, sin etiquetas, estandarizados

Se quitan las 1177 filas idénticas (módulo 4, notebook 02: en este dataset **sí** son
duplicados) y se apartan `tipo` y `quality`. No las usa ningún algoritmo; quedan para
validar al final.

In [ ]:
vinos = pd.read_csv("../datos/wine-quality.csv").drop_duplicates().reset_index(drop=True)
X = vinos.drop(columns=["quality", "tipo"])
tipo = vinos["tipo"]
calidad = vinos["quality"]
print(f"{len(vinos)} vinos sin duplicados · {X.shape[1]} variables · "
      f"{(tipo == 'tinto').sum()} tintos, {(tipo == 'blanco').sum()} blancos")

print("\nVarianza de cada variable en sus unidades originales:")
print(X.var().round(3).sort_values(ascending=False).to_string())

`total_sulfur_dioxide` tiene una varianza de 3223; `density`, de 0.00001. Una distancia
euclídea sobre estas columnas **es** la distancia en dióxido de azufre: el resto no
cuenta. K-Means, el jerárquico y DBSCAN trabajan con distancias, así que estandarizar no
es opcional. Lo medimos con la etiqueta que el algoritmo no ve:

In [ ]:
escalador = StandardScaler()
X_esc = escalador.fit_transform(X)
etiq_tipo = (tipo == "tinto").astype(int).to_numpy()

for nombre, datos in [("sin estandarizar", X.to_numpy()), ("estandarizado", X_esc)]:
    km = KMeans(n_clusters=2, n_init=10, random_state=SEMILLA).fit(datos)
    print(f"K-Means (k=2) {nombre:<17} ARI contra `tipo`: {adjusted_rand_score(etiq_tipo, km.labels_):.3f}")

## 2. Elegir $k$: tres criterios y una referencia nula

Para $k$ de 2 a 10, con K-Means++ y 10 inicializaciones: inercia (codo), silueta media, e
índice de **Davies-Bouldin** (promedio, sobre cada grupo, del cociente entre la dispersión
de los dos grupos más parecidos y la distancia entre sus centroides; **menor es mejor**).
Los tres de `02-validacion-clusters.md`.

Y una **referencia nula**: los mismos datos con cada columna permutada de forma
independiente. Conserva la distribución de cada variable, pero destruye toda relación
entre ellas — es decir, no hay grupos por construcción. Si la silueta de los datos reales
no se separa de la de la referencia, lo que K-Means "encontró" es lo que K-Means impone
(notebook 01, sección 7).

In [ ]:
X_nulo = np.column_stack([rng.permutation(X_esc[:, j]) for j in range(X_esc.shape[1])])
ks = range(2, 11)
filas = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(X_esc)
    km_nulo = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(X_nulo)
    filas.append({"k": k, "inercia": km.inertia_, "silueta": silhouette_score(X_esc, km.labels_),
                  "Davies-Bouldin": davies_bouldin_score(X_esc, km.labels_),
                  "silueta (nulo)": silhouette_score(X_nulo, km_nulo.labels_)})
criterios = pd.DataFrame(filas).set_index("k")
print(criterios.round(3).to_string())

fig, ejes = plt.subplots(1, 3, figsize=(16, 4))
ejes[0].plot(ks, criterios["inercia"], "o-")
ejes[0].set_title("Inercia (codo)")
ejes[1].plot(ks, criterios["silueta"], "o-", label="Wine Quality")
ejes[1].plot(ks, criterios["silueta (nulo)"], "s--", color="gray", label="referencia nula (columnas permutadas)")
ejes[1].set_title("Silueta (mayor es mejor)")
ejes[1].legend()
ejes[2].plot(ks, criterios["Davies-Bouldin"], "o-")
ejes[2].set_title("Davies-Bouldin (menor es mejor)")
for eje in ejes:
    eje.set_xlabel("k")
plt.show()

Tres lecturas, no del todo coincidentes:

- La **inercia** no tiene un codo limpio: baja mucho de 2 a 3, algo menos de 3 a 4 y
  luego poco a poco. Es lo habitual con datos reales; el codo se ve claro solo en los
  datos sintéticos del notebook 01.
- La **silueta** tiene su máximo en $k = 2$ (0.27) y vuelve a subir un poco en $k = 4$.
- **Davies-Bouldin** prefiere $k = 4$ (1.48) a $k = 2$ (1.63).

Los criterios miden cosas distintas —la silueta, separación relativa punto a punto;
Davies-Bouldin, solapamiento entre pares de centroides— y no hay razón para que
coincidan. Lo que sí es concluyente es la referencia nula: la silueta de los datos reales
(0.27 en $k=2$) casi triplica la de las columnas permutadas (0.10): **hay estructura**.
Pero 0.27 es, en la escala de Kaufman y Rousseeuw, "estructura débil": grupos que se
tocan, no islas. Empezamos por $k = 2$, que es el más claro, y volvemos a $k = 4$ después.

## 3. Qué encontró K-Means con $k = 2$

Un grupo no se interpreta mirando centroides estandarizados: se vuelve a las **unidades
originales** y se compara el perfil de cada grupo con el global.

In [ ]:
km2 = KMeans(n_clusters=2, n_init=10, random_state=SEMILLA).fit(X_esc)
grupo2 = km2.labels_

perfil = X.groupby(grupo2).mean().T
perfil.columns = [f"grupo {g} (n={n})" for g, n in zip(perfil.columns, np.bincount(grupo2))]
perfil["cociente 1/0"] = perfil.iloc[:, 1] / perfil.iloc[:, 0]
print(perfil.round(3).to_string())

El grupo 1 tiene el doble de acidez volátil y de cloruros, un tercio del dióxido de
azufre total, menos de la mitad de azúcar residual y más sulfatos. Cualquiera que sepa de
vino lo reconoce: es la química de un **tinto** frente a un **blanco** (los blancos se
sulfitan más y suelen ser más dulces). K-Means no vio la columna `tipo`, pero la
diferencia tinto/blanco es la **dirección de mayor varianza** de estos datos, y eso es lo
que K-Means encuentra primero. La validación *a posteriori* lo confirma:

In [ ]:
tabla = pd.crosstab(tipo, grupo2, rownames=["tipo real"], colnames=["grupo K-Means"])
print(tabla.to_string())
print(f"\nARI contra `tipo`: {adjusted_rand_score(etiq_tipo, grupo2):.3f}   "
      f"vinos en el grupo 'equivocado': {tabla.iloc[0, 1] + tabla.iloc[1, 0]} de {len(vinos)} "
      f"({100 * (tabla.iloc[0, 1] + tabla.iloc[1, 0]) / len(vinos):.1f} %)")

### Lo que **no** encontró: la calidad

La pregunta de negocio del módulo 4 era predecir la calidad. ¿La recupera el clustering?
Medimos con la **información mutua normalizada** (NMI) entre la partición y `quality`
(0 = independientes, 1 = una determina la otra), para cada $k$:

In [ ]:
nmi = {}
for k in ks:
    etiq = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(X_esc).labels_
    nmi[k] = {"NMI con `tipo`": normalized_mutual_info_score(etiq_tipo, etiq),
              "NMI con `quality`": normalized_mutual_info_score(calidad, etiq)}
print(pd.DataFrame(nmi).T.round(3).to_string())

Con ningún $k$ los grupos dicen algo de la calidad (NMI $\leq 0.07$; con `tipo`, 0.86 en
$k=2$). No es un fallo: el clustering encuentra la estructura **dominante** de $\mathbf{X}$,
que aquí es tinto/blanco, y la calidad es una señal sutil repartida en varias variables
(el módulo 4 necesitó un ensamble afinado para llegar a AP 0.59). **Clustering no es
clasificación sin etiquetas**: no hay ninguna garantía de que los grupos que existen sean
los que a uno le interesan. Si lo que se quiere es predecir la calidad, hay que
etiquetar.

## 4. Segundo nivel: ¿hay grupos dentro de los blancos?

Tinto/blanco es tan dominante que aplasta cualquier otra estructura. Repetimos el análisis
**solo con los blancos** (3961 vinos), reestandarizando, con su propia referencia nula.

In [ ]:
mascara_b = (tipo == "blanco").to_numpy()
X_b = StandardScaler().fit_transform(X[mascara_b])
X_b_nulo = np.column_stack([rng.permutation(X_b[:, j]) for j in range(X_b.shape[1])])

filas = []
for k in range(2, 7):
    km_b = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(X_b)
    km_bn = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(X_b_nulo)
    filas.append({"k": k, "silueta": silhouette_score(X_b, km_b.labels_),
                  "silueta (nulo)": silhouette_score(X_b_nulo, km_bn.labels_),
                  "Davies-Bouldin": davies_bouldin_score(X_b, km_b.labels_)})
print(pd.DataFrame(filas).set_index("k").round(3).to_string())

km_b2 = KMeans(n_clusters=2, n_init=10, random_state=SEMILLA).fit(X_b)
perfil_b = X[mascara_b].groupby(km_b2.labels_).mean().T
perfil_b.columns = [f"blancos {g} (n={n})" for g, n in zip(perfil_b.columns, np.bincount(km_b2.labels_))]
perfil_b["cociente 1/0"] = perfil_b.iloc[:, 1] / perfil_b.iloc[:, 0]
print("\n" + perfil_b.round(3).to_string())
print(f"\nCalidad media por grupo: {calidad[mascara_b].groupby(km_b2.labels_).mean().round(2).to_dict()}")

Otra vez $k = 2$, con silueta 0.21 frente a 0.08 de la referencia nula: estructura real,
débil. Y otra vez interpretable: el grupo 1 tiene **tres veces más azúcar residual**, más
densidad, más dióxido de azufre y 1.5 grados menos de alcohol — blancos **dulces**
(el azúcar que no fermentó no se convirtió en alcohol) frente a **secos**. Los dulces
tienen calidad media algo menor (5.54 frente a 6.04), la primera relación con la calidad
que aparece, y es indirecta: `alcohol` es la variable más correlacionada con `quality`
(módulo 4).

Los $k = 4$ del conjunto completo que prefería Davies-Bouldin son, aproximadamente, esta
jerarquía aplanada: blancos secos, blancos dulces, y los tintos partidos en dos:

In [ ]:
grupo4 = KMeans(n_clusters=4, n_init=10, random_state=SEMILLA).fit(X_esc).labels_
dulce = int(np.argmax(X[mascara_b].groupby(km_b2.labels_)["residual_sugar"].mean()))
subgrupo = np.full(len(vinos), "tinto", dtype=object)
subgrupo[mascara_b] = np.where(km_b2.labels_ == dulce, "blanco dulce", "blanco seco")
print(pd.crosstab(subgrupo, grupo4, rownames=["tipo / subgrupo"], colnames=["grupo K-Means (k=4)"]).to_string())

## 5. Jerárquico: Ward coincide con K-Means; los demás *linkages*, no

Con 5320 filas la matriz de distancias cabe en memoria (5320² × 8 bytes ≈ 226 MB), así que
el jerárquico es viable. Cortamos en dos grupos con los cuatro *linkages*:

In [ ]:
filas = []
for metodo in ["ward", "average", "complete", "single"]:
    etiq = AgglomerativeClustering(n_clusters=2, linkage=metodo).fit(X_esc).labels_
    filas.append({"linkage": metodo, "tamaños": np.bincount(etiq).tolist(),
                  "ARI con K-Means (k=2)": adjusted_rand_score(grupo2, etiq),
                  "ARI con `tipo`": adjusted_rand_score(etiq_tipo, etiq)})
print(pd.DataFrame(filas).round(3).to_string(index=False))

**Ward** —que fusiona los dos grupos cuya unión aumenta menos la inercia, el criterio de
K-Means— llega casi a la misma partición (ARI 0.90). Los otros tres cortan **un solo vino**
y dejan los 5319 restantes juntos. No es un error: en datos reales hay puntos atípicos, y
el punto más lejano de todos se fusiona el último con cualquier *linkage* basado en
distancias entre puntos. Para llegar a grupos útiles habría que cortar mucho más abajo, o
quitar los atípicos primero. El dendrograma truncado de Ward muestra la jerarquía: la
primera división es tinto/blanco, y la siguiente parte los blancos.

In [ ]:
Z = linkage(X_esc, method="ward")
fig, eje = plt.subplots(figsize=(11, 4))
dendrogram(Z, truncate_mode="lastp", p=12, ax=eje, leaf_font_size=9, show_contracted=True)
eje.set_title("Dendrograma de Ward, truncado a las últimas 12 fusiones (entre paréntesis: vinos por rama)")
eje.set_ylabel("Aumento de inercia al fusionar")
plt.show()
print("Alturas de las últimas cuatro fusiones:", np.round(Z[-4:, 2], 1))

## 6. DBSCAN en 11 dimensiones: más detector de anomalías que de grupos

En el notebook 01 DBSCAN resolvió las medias lunas. En 11 dimensiones se enfrenta a un
problema distinto: las distancias entre puntos **se concentran** —la distancia al vecino
más cercano y al más lejano se parecen cada vez más a medida que crece la dimensión—, así
que no hay una ventana clara de `eps` que separe "denso" de "vacío". El gráfico de
k-distancias lo muestra: no hay codo, sino una pendiente suave con una cola.

In [ ]:
MIN_SAMPLES = 10
d_k = np.sort(NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_esc).kneighbors(X_esc)[0][:, -1])[::-1]

filas = []
for eps in [1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]:
    etiq = DBSCAN(eps=eps, min_samples=MIN_SAMPLES).fit(X_esc).labels_
    con_grupo = etiq >= 0
    filas.append({"eps": eps, "grupos": etiq.max() + 1, "% ruido": 100 * np.mean(~con_grupo),
                  "tamaño del mayor": np.bincount(etiq[con_grupo]).max() if con_grupo.any() else 0,
                  "ARI con `tipo` (sin ruido)": adjusted_rand_score(etiq_tipo[con_grupo], etiq[con_grupo])})
print(pd.DataFrame(filas).round(3).to_string(index=False))

fig, eje = plt.subplots(figsize=(7, 4))
eje.plot(d_k)
eje.set_xlabel("Vinos, ordenados por distancia a su 10.º vecino")
eje.set_ylabel("Distancia al 10.º vecino (estandarizada)")
eje.set_title("k-distancias en 11 dimensiones: sin codo claro")
plt.show()

Con `eps = 1`, el 71 % es ruido y los "grupos" son fragmentos; con `eps ≥ 1.75`, todo es
un solo grupo. Hay un punto intermedio interesante: con `eps = 1.25` DBSCAN separa
tinto/blanco casi perfectamente (ARI 0.98) **entre los vinos que no son ruido** — los
núcleos densos de cada tipo—, pero a costa de dejar fuera a un tercio; con `eps = 1.5`
los dos núcleos ya se tocan y se funden. No hay un `eps` que dé los dos grupos completos,
porque en 11-D las fronteras entre regiones no están vacías. Lo que sí hace bien es lo
otro: con `eps = 2`, señala **191 vinos (3.6 %) como ruido** — puntos sin 10 vecinos a
distancia 2 en el espacio estandarizado. ¿Qué son?

In [ ]:
etiq_db = DBSCAN(eps=2.0, min_samples=MIN_SAMPLES).fit(X_esc).labels_
ruido = etiq_db == -1
z_max = np.abs(X_esc).max(axis=1)
print(f"Vinos marcados como ruido: {ruido.sum()}")
print(f"Mediana del |z| máximo por vino — ruido: {np.median(z_max[ruido]):.2f}   resto: {np.median(z_max[~ruido]):.2f}")
print(f"Vinos con alguna variable a más de 4 desviaciones — ruido: {100 * np.mean(z_max[ruido] > 4):.0f} %   "
      f"resto: {100 * np.mean(z_max[~ruido] > 4):.1f} %")
print(f"Proporción de vinos buenos (quality ≥ 7) — ruido: {100 * (calidad[ruido] >= 7).mean():.1f} %   "
      f"resto: {100 * (calidad[~ruido] >= 7).mean():.1f} %")

extremos = X[ruido].agg(["min", "max"]).T
extremos["min global"] = X.min()
extremos["max global"] = X.max()
print("\nRango de los vinos-ruido frente al rango global (las columnas con el extremo global entre el ruido):")
print(extremos[(extremos["min"] == extremos["min global"]) | (extremos["max"] == extremos["max global"])].round(3).to_string())

El ruido de DBSCAN son vinos con **al menos una variable extrema**: la mediana de su
$|z|$ máximo es 4.0, contra 1.65 en el resto; y el máximo global de **las once
variables** está entre ellos. Son también peores: 6.8 % de
buenos frente a 19.4 %. En un flujo real, esta lista es lo que se manda a revisar antes
de entrenar nada: errores de medición, vinos de otra categoría (los 65.8 g/L de azúcar
son un vino de postre), o casos genuinamente raros que el modelo del módulo 4 no va a
predecir bien.

### 🔵 Isolation Forest: otra definición de "raro"

**Isolation Forest** (Liu et al., 2008) aísla puntos con particiones aleatorias: un punto
atípico queda solo en pocas particiones; uno normal necesita muchas. No usa distancias ni
densidad, así que no sufre la concentración de distancias. Le pedimos la misma fracción de
anomalías que marcó DBSCAN y comparamos las dos listas:

In [ ]:
bosque = IsolationForest(contamination=ruido.mean(), random_state=SEMILLA).fit(X_esc)
anomalia_if = bosque.predict(X_esc) == -1
coinciden = anomalia_if & ruido
print(f"Isolation Forest marca {anomalia_if.sum()} vinos; DBSCAN, {ruido.sum()}; en común: {coinciden.sum()} "
      f"(Jaccard {coinciden.sum() / (anomalia_if | ruido).sum():.2f})")
print(f"Mediana del |z| máximo — solo IF: {np.median(z_max[anomalia_if & ~ruido]):.2f}   "
      f"solo DBSCAN: {np.median(z_max[ruido & ~anomalia_if]):.2f}   ambos: {np.median(z_max[coinciden]):.2f}")

Coinciden en un tercio. Los 95 que señalan ambos son los más extremos ($|z|$ máximo 4.7);
los que señala solo uno de los dos tienen extremos más moderados (≈3.3) y son "raros" por
razones distintas: para DBSCAN, estar lejos de todo; para Isolation Forest, ser fácil de
aislar con cortes por una variable. "Anomalía" no tiene una definición única: cada detector la define con su
algoritmo, y la lista correcta es la que sirve para lo que se va a hacer con ella.

## 7. Estabilidad: ¿la partición depende de la muestra?

Un último diagnóstico, y el más barato: si los grupos son reales, deberían reaparecer al
entrenar sobre **la mitad de los datos**. Para cada $k$, 10 submuestras aleatorias del
50 %, K-Means sobre cada una, y ARI contra la partición del conjunto completo restringida
a la submuestra.

In [ ]:
def estabilidad(datos, k, n_rep=10):
    completo = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(datos).labels_
    aris = []
    for r in range(n_rep):
        idx = rng.choice(len(datos), len(datos) // 2, replace=False)
        parcial = KMeans(n_clusters=k, n_init=10, random_state=r).fit(datos[idx]).labels_
        aris.append(adjusted_rand_score(completo[idx], parcial))
    return np.mean(aris), np.std(aris, ddof=1) / np.sqrt(n_rep)


filas = []
for k in range(2, 8):
    media, ee = estabilidad(X_esc, k)
    filas.append({"k": k, "ARI medio (submuestra vs. completo)": media, "ee": ee})
print(pd.DataFrame(filas).set_index("k").round(3).to_string())

$k = 2$, 3 y 4 son estables (ARI > 0.97): los mismos grupos aparecen con la mitad de los
datos. A partir de $k = 5$ la estabilidad cae: las particiones adicionales dependen de qué
vinos entran en la muestra, que es la firma de estar cortando un continuo por donde
caiga. Es el mismo criterio de "detectable frente a ruido" del módulo 3, aplicado a un
problema sin $y$.

## Resumen

| Pregunta | Respuesta medida |
|---|---|
| ¿Estandarizar? | Obligatorio: sin estandarizar, la distancia es dióxido de azufre y el ARI contra `tipo` cae de 0.93 a 0.31 |
| ¿Cuántos grupos? | Silueta dice 2, Davies-Bouldin dice 4, la inercia no tiene codo; la referencia nula confirma que hay estructura (0.27 frente a 0.10), débil |
| ¿Qué son los grupos? | $k=2$: tinto/blanco (ARI 0.93, 1.6 % de vinos cruzados), sin haber visto `tipo`; dentro de los blancos, dulces/secos |
| ¿Y la calidad? | NMI $\leq 0.07$ con cualquier $k$: el clustering encuentra la estructura dominante, no la que interesa |
| ¿Jerárquico? | Ward ≈ K-Means (ARI 0.90); *average*, *complete* y *single* aíslan un solo vino atípico |
| ¿DBSCAN? | En 11-D solo separa tinto/blanco dejando un tercio como ruido (distancias concentradas); con `eps = 2` señala 191 vinos extremos, un tercio de ellos en común con Isolation Forest |
| ¿Estable? | $k \leq 4$: ARI > 0.97 con la mitad de los datos; $k \geq 5$: cae |